# CodiEsp Medical Coding — Kaggle Training

Trains both stages of the pipeline on a T4 GPU:
- **Stage 1 — NER**: Extracts clinical spans from Spanish text (bsc-bio-ehr-es fine-tuned)
- **Stage 2 — Reranker**: Maps extracted spans to ICD-10 codes (cross-encoder)
- **Stage 3 — Inference**: Runs the full pipeline on the test set and evaluates

**Before running:** `Runtime > Change runtime type > T4 GPU`

Then **Runtime > Run all** — everything else is automated.

## 1. Check GPU

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU detected!\n"
        "Go to: Runtime > Change runtime type > Hardware accelerator > T4 GPU"
    )

gpu_name = torch.cuda.get_device_name(0)
vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU  : {gpu_name}")
print(f"VRAM : {vram_gb:.1f} GB")

## 2. Clone Repository

In [24]:
import os

REPO_URL = "https://github.com/tylerklement/medical-coding.git"
REPO_DIR = "/content/medical-coding"

if os.path.isdir(REPO_DIR):
    print("Repo already cloned — pulling latest changes...")
    !git -C {REPO_DIR} pull
else:
    print(f"Cloning {REPO_URL}...")
    !git clone {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")
print("Contents:", sorted(os.listdir('.')))

Repo already cloned — pulling latest changes...
remote: Enumerating objects: 9, done.
remote: Counting objects: 100% (9/9), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 5 (delta 4), reused 5 (delta 4), pack-reused 0 (from 0)
Unpacking objects: 100% (5/5), 1.87 KiB | 956.00 KiB/s, done.
From https://github.com/tylerklement/medical-coding
   bff5dce..1ad861c  master     -> origin/master
Updating bff5dce..1ad861c
Fast-forward
 src/code_mapper.py | 79 +++++++++++++++++++++++++++++++++++++++++++++++++-----
 src/pipeline.py    | 54 ++++++++++++++++++++-----------------
 2 files changed, 101 insertions(+), 32 deletions(-)
Working directory: /kaggle/working/medical-coding
Contents: ['.git', '.gitignore', 'checkpoints', 'data', 'models', 'notebooks', 'outputs', 'predict.py', 'requirements.txt', 'scripts', 'src', 'train_ner.py', 'train_reranker.py']


## 3. Install Dependencies

In [25]:
!pip install -q "transformers>=4.46" datasets seqeval sentence-transformers faiss-cpu accelerate
print("Dependencies ready.")

Dependencies ready.


## 4. Mount Drive & Set Up Cache
Data and models are cached in Google Drive so future runs skip downloads.

In [26]:

import shutil, json
from pathlib import Path

CACHE_DIR = Path('/kaggle/working/medical-coding-cache')
DATA_DIR    = Path('data')
CACHE_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)

def ensure_data():
    """Restore CodiEsp data + ICD-10 index from Cache cache if missing locally."""
    cache_data = CACHE_DIR / 'data'
    local_tsv  = DATA_DIR / 'train' / 'trainX.tsv'
    local_cm   = DATA_DIR / 'icd10cm_descriptions.json'
    local_pcs  = DATA_DIR / 'icd10pcs_descriptions.json'

    if not local_tsv.exists():
        if (cache_data / 'train' / 'trainX.tsv').exists():
            print("Restoring CodiEsp data from Cache...")
            if DATA_DIR.exists(): shutil.rmtree(DATA_DIR)
            shutil.copytree(cache_data, DATA_DIR)
        else:
            print("Downloading CodiEsp from Zenodo...")
            os.system('python scripts/download_data.py')
            if cache_data.exists(): shutil.rmtree(cache_data)
            shutil.copytree(DATA_DIR, cache_data)
        print("  CodiEsp data ready.")
    else:
        print("  CodiEsp data already present.")

    cache_cm  = CACHE_DIR / 'icd10cm_descriptions.json'
    cache_pcs = CACHE_DIR / 'icd10pcs_descriptions.json'
    if not local_cm.exists():
        if cache_cm.exists() and cache_cm.stat().st_size > 10_000:
            print("Restoring ICD-10 index from Cache...")
            shutil.copy(cache_cm,  local_cm)
            shutil.copy(cache_pcs, local_pcs)
        else:
            print("Building ICD-10 index from CMS...")
            os.system('python scripts/build_icd10_index.py')
            shutil.copy(local_cm,  cache_cm)
            shutil.copy(local_pcs, cache_pcs)
        print(f"  CM: {len(json.load(open(local_cm))):,} codes  |  PCS: {len(json.load(open(local_pcs))):,} codes")
    else:
        print("  ICD-10 index already present.")

ensure_data()
print("All data ready.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
  CodiEsp data already present.
  ICD-10 index already present.
All data ready.


---
## Stage 1: NER Training
Fine-tunes `bsc-bio-ehr-es` to extract clinical spans from Spanish text.

**T4 estimate: ~25-40 min for 5 epochs on the full dataset.**

In [ ]:
ensure_data()

NER_EPOCHS     = 5
NER_BATCH      = 16
NER_GRAD_ACCUM = 1
NER_LR         = 2e-5
NER_WORKERS    = 2
NER_OUT        = 'models/ner'
NER_MAX_TRAIN  = None   # set to int for smoke-test, e.g. 50
NER_MAX_DEV    = None

import os
os.makedirs(NER_OUT, exist_ok=True)
os.makedirs('outputs', exist_ok=True)

ner_args = (
    f"--no_wandb --fp16"
    f" --epochs {NER_EPOCHS}"
    f" --batch_size {NER_BATCH}"
    f" --grad_accum {NER_GRAD_ACCUM}"
    f" --learning_rate {NER_LR}"
    f" --num_workers {NER_WORKERS}"
    f" --output_dir {NER_OUT}"
)
if NER_MAX_TRAIN: ner_args += f" --max_train_samples {NER_MAX_TRAIN}"
if NER_MAX_DEV:   ner_args += f" --max_dev_samples {NER_MAX_DEV}"

print("Command: python train_ner.py", ner_args)
print("=" * 70)
!python train_ner.py {ner_args}

In [ ]:
import json, shutil
from pathlib import Path

cache_ner = CACHE_DIR / 'models' / 'ner'
local_ner = Path(NER_OUT)

if local_ner.exists():
    cache_ner.parent.mkdir(parents=True, exist_ok=True)
    if cache_ner.exists(): shutil.rmtree(cache_ner)
    shutil.copytree(local_ner, cache_ner)
    print(f"NER model saved to Cache: {cache_ner}")

results_path = Path('outputs/ner_results.json')
if results_path.exists():
    r = json.load(open(results_path))
    print("\nNER dev-set results:")
    print(f"  Precision : {r.get('eval_precision', 0):.4f}")
    print(f"  Recall    : {r.get('eval_recall',    0):.4f}")
    print(f"  F1        : {r.get('eval_f1',        0):.4f}")

---
## Stage 2: Reranker Training
Fine-tunes a cross-encoder to score `(clinical span, ICD-10 description)` pairs.
Hard negatives are mined with a bi-encoder, making it learn subtle code distinctions.

**T4 estimate: ~5-10 min for 3 epochs.**

In [ ]:
ensure_data()

RR_EPOCHS    = 3
RR_BATCH     = 32
RR_NEGATIVES = 4
RR_OUT       = 'models/reranker'

import os
os.makedirs(RR_OUT, exist_ok=True)

rr_args = (
    f"--no_wandb"
    f" --epochs {RR_EPOCHS}"
    f" --batch_size {RR_BATCH}"
    f" --negatives_per_pos {RR_NEGATIVES}"
    f" --output_dir {RR_OUT}"
)

print("Command: python train_reranker.py", rr_args)
print("=" * 70)
!python train_reranker.py {rr_args}

In [ ]:
import shutil
from pathlib import Path

cache_rr = CACHE_DIR / 'models' / 'reranker'
local_rr = Path(RR_OUT)

if local_rr.exists():
    cache_rr.parent.mkdir(parents=True, exist_ok=True)
    if cache_rr.exists(): shutil.rmtree(cache_rr)
    shutil.copytree(local_rr, cache_rr)
    print(f"Reranker model saved to Cache: {cache_rr}")
    print(f"Files: {sorted(f.name for f in cache_rr.iterdir())}")
else:
    print("Reranker directory not found — did training complete?")

---
## Stage 3: Inference on Test Set
Runs the full pipeline (NER → FAISS retrieval → cross-encoder reranking) on every
document in the test split, then evaluates against gold labels.

**T4 estimate: ~10-20 min for all 250 test documents.**

In [27]:
ensure_data()

import os, shutil
from pathlib import Path

NER_MODEL_PATH = 'models/ner'
RERANKER_PATH  = 'models/reranker'
FAISS_CACHE    = CACHE_DIR / 'faiss'

# Restore NER model from Cache if not present
cache_ner = CACHE_DIR / 'models' / 'ner'
if not Path(NER_MODEL_PATH).exists() and cache_ner.exists():
    shutil.copytree(cache_ner, NER_MODEL_PATH)
    print('NER model restored from Cache.')
elif Path(NER_MODEL_PATH).exists():
    print('NER model already present.')
else:
    raise FileNotFoundError('NER model not found — run Stage 1 training first.')

# Restore reranker from Cache if not present
cache_rr = CACHE_DIR / 'models' / 'reranker'
if not Path(RERANKER_PATH).exists() and cache_rr.exists():
    shutil.copytree(cache_rr, RERANKER_PATH)
    print('Reranker restored from Cache.')
elif Path(RERANKER_PATH).exists():
    print('Reranker already present.')
else:
    raise FileNotFoundError('Reranker not found — run Stage 2 training first.')

# Restore FAISS index from Cache (avoids ~3 min rebuild on re-runs)
local_faiss = Path('models/faiss_cm.index')
cache_faiss = FAISS_CACHE / 'faiss_cm.index'
if not local_faiss.exists() and cache_faiss.exists():
    print('Restoring FAISS index from Cache...')
    for fname in ['faiss_cm.index', 'faiss_pcs.index', 'cm_codes.pkl', 'pcs_codes.pkl']:
        src = FAISS_CACHE / fname
        dst = Path('models') / fname
        if src.exists(): shutil.copy(src, dst)
    print('FAISS index restored.')
elif local_faiss.exists():
    print('FAISS index already present.')
else:
    print('FAISS index will be built on first pipeline load (~3 min).')

print('Ready for inference.')

  CodiEsp data already present.
  ICD-10 index already present.
NER model already present.
Reranker already present.
FAISS index already present.
Ready for inference.


In [28]:
os.makedirs('outputs/predictions', exist_ok=True)

predict_args = (
    f"--input data/test/text_files"
    f" --output outputs/predictions"
    f" --ner_model {NER_MODEL_PATH}"
    f" --cross_encoder {RERANKER_PATH}"
    f" --format tsv"
    f" --threshold 0.55"
)

print("Command: python predict.py", predict_args)
print("=" * 70)
!python predict.py {predict_args}

# Cache FAISS index to Cache after first build so future runs skip it
if not cache_faiss.exists():
    print('Caching FAISS index to Cache...')
    FAISS_CACHE.mkdir(parents=True, exist_ok=True)
    for fname in ['faiss_cm.index', 'faiss_pcs.index', 'cm_codes.pkl', 'pcs_codes.pkl']:
        src = Path('models') / fname
        if src.exists(): shutil.copy(src, FAISS_CACHE / fname)
    print('FAISS index cached to Cache.')

Command: python predict.py --input data/test/text_files --output outputs/predictions --ner_model models/ner --cross_encoder models/reranker --format tsv --threshold 0.4
── Loading NER model ────────────────────────────────────
Loading weights: 100% 199/199 [00:00<00:00, 1397.58it/s, Materializing param=roberta.encoder.layer.11.output.dense.weight]
── Loading Code Mapper ──────────────────────────────────
Loading bi-encoder …
Loading weights: 100% 199/199 [00:00<00:00, 929.80it/s, Materializing param=pooler.dense.weight]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading cross-encoder …
Loading weights: 100% 105/105 [00:00<00:00, 1231.67it/s, Materializing param=classifier.weight]
Loading ICD-10-C

In [29]:
# Evaluate predictions against gold test labels
import pandas as pd
from pathlib import Path

pred_tsv = 'outputs/predictions/predictions.tsv'
gold_tsv = 'data/test/testX.tsv'

if not Path(pred_tsv).exists():
    print('No predictions found — did inference complete?')
else:
    pred = pd.read_csv(pred_tsv, sep='\t', header=None,
                       names=['article_id','label','code','span','pos'])
    gold = pd.read_csv(gold_tsv, sep='\t', header=None,
                       names=['article_id','label','code','span','pos'])

    # Micro-averaged P/R/F1 over all articles
    all_tp = all_fp = all_fn = 0
    pred_ids = set(pred['article_id'].unique())
    for art_id in gold['article_id'].unique():
        gold_codes = set(gold[gold.article_id == art_id]['code'].str.lower())
        pred_codes = set(pred[pred.article_id == art_id]['code'].str.lower()) if art_id in pred_ids else set()
        all_tp += len(gold_codes & pred_codes)
        all_fp += len(pred_codes - gold_codes)
        all_fn += len(gold_codes - pred_codes)

    precision = all_tp / (all_tp + all_fp) if (all_tp + all_fp) > 0 else 0
    recall    = all_tp / (all_tp + all_fn) if (all_tp + all_fn) > 0 else 0
    f1        = 2*precision*recall/(precision+recall) if (precision+recall) > 0 else 0

    print('=== End-to-End Test Results ===')
    print(f'  Precision : {precision:.4f}')
    print(f'  Recall    : {recall:.4f}')
    print(f'  F1        : {f1:.4f}')
    print(f'  Total predictions : {len(pred):,}')
    print(f'  Total gold labels : {len(gold):,}')
    print()
    print('Sample predictions:')
    print(pred.head(10).to_string(index=False))

=== End-to-End Test Results ===
  Precision : 0.0024
  Recall    : 0.0075
  F1        : 0.0037
  Total predictions : 34,344
  Total gold labels : 4,777

Sample predictions:
               article_id       label    code              span     pos
S0004-06142005000500011-1 DIAGNOSTICO   m0219                 a 128 129
S0004-06142005000500011-1 DIAGNOSTICO   m0218                 c 129 130
S0004-06142005000500011-1 DIAGNOSTICO   m0218                 c 130 131
S0004-06142005000500011-1 DIAGNOSTICO  m02152                 i 131 132
S0004-06142005000500011-1 DIAGNOSTICO    g729                 d 132 133
S0004-06142005000500011-1 DIAGNOSTICO    i252                 e 133 134
S0004-06142005000500011-1 DIAGNOSTICO     f04                 n 134 135
S0004-06142005000500011-1 DIAGNOSTICO   m0218                 t 135 136
S0004-06142005000500011-1 DIAGNOSTICO  m02141 e laboral antiguo 136 153
S0004-06142005000500011-1 DIAGNOSTICO t465x6d                 f 158 159
